# Customer Feedback Sentiment Analysis
### Exploratory Data Analysis & Visualization

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from collections import Counter
from wordcloud import WordCloud
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from utils.preprocessor import preprocess_text
from utils.analyzer import SentimentAnalyzer

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
%matplotlib inline
print('Libraries loaded ✓')

## 1. Load & Inspect Dataset

In [ ]:
# Replace with your real CSV path
# df = pd.read_csv('../data/reviews.csv')

# Sample data for demo
sample_reviews = [
    {'text': 'Absolutely love this product!', 'sentiment': 'positive', 'rating': 5},
    {'text': 'Great customer support, very helpful.', 'sentiment': 'positive', 'rating': 5},
    {'text': 'Delivery was too slow, very disappointed.', 'sentiment': 'negative', 'rating': 1},
    {'text': 'Product broke after two days.', 'sentiment': 'negative', 'rating': 1},
    {'text': 'Average product, nothing special.', 'sentiment': 'neutral', 'rating': 3},
    {'text': 'It works fine, meets expectations.', 'sentiment': 'neutral', 'rating': 3},
    {'text': 'Best purchase I made this year!', 'sentiment': 'positive', 'rating': 5},
    {'text': 'App crashes frequently, very frustrating.', 'sentiment': 'negative', 'rating': 2},
    {'text': 'Decent value for the price.', 'sentiment': 'neutral', 'rating': 3},
    {'text': 'Exceeded all expectations. Highly recommend!', 'sentiment': 'positive', 'rating': 5},
]

df = pd.DataFrame(sample_reviews)
print(df.shape)
df.head()

## 2. Sentiment Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

colors = {'positive': '#1D9E75', 'negative': '#D85A30', 'neutral': '#378ADD'}
counts = df['sentiment'].value_counts()

# Bar chart
bars = axes[0].bar(counts.index, counts.values,
                   color=[colors[s] for s in counts.index], edgecolor='white', linewidth=1.5)
axes[0].set_title('Sentiment Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count')
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 str(val), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(counts.values, labels=counts.index,
            colors=[colors[s] for s in counts.index],
            autopct='%1.1f%%', startangle=90, pctdistance=0.8)
axes[1].set_title('Sentiment Share', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../static/sentiment_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Text Preprocessing Pipeline

In [ ]:
df['tokens'] = df['text'].apply(preprocess_text)
df['token_count'] = df['tokens'].apply(len)

print('Avg tokens per review:')
print(df.groupby('sentiment')['token_count'].mean().round(2))
df[['text', 'tokens', 'token_count', 'sentiment']].head()

## 4. Word Clouds by Sentiment

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
wc_colors = {'positive': '#1D9E75', 'negative': '#D85A30', 'neutral': '#378ADD'}

for ax, sentiment in zip(axes, ['positive', 'negative', 'neutral']):
    subset = df[df['sentiment'] == sentiment]['tokens'].explode()
    if len(subset) == 0:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center')
    else:
        freq = Counter(subset)
        wc = WordCloud(width=400, height=200, background_color='white',
                       color_func=lambda *args, **kwargs: wc_colors[sentiment],
                       max_words=30).generate_from_frequencies(freq)
        ax.imshow(wc, interpolation='bilinear')
    ax.set_title(sentiment.capitalize(), fontsize=13, fontweight='bold',
                 color=wc_colors[sentiment])
    ax.axis('off')

plt.suptitle('Word Clouds by Sentiment', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../static/wordclouds.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Real-time Predictions

In [ ]:
analyzer = SentimentAnalyzer()

test_reviews = [
    'This product is absolutely wonderful, exceeded all my expectations!',
    'Horrible experience, the item arrived broken and support was useless.',
    'The product is okay, nothing to complain about but nothing special.',
]

for review in test_reviews:
    tokens = preprocess_text(review)
    result = analyzer.predict(review, tokens)
    print(f"Review : {review[:60]}...")
    print(f"Sentiment : {result['sentiment'].upper()} ({result['confidence_pct']})")
    print(f"Scores  : {result['scores']}")
    print()